# 🏗️ Scenario 3 — Explorable 3D World from a 2D Image · Kaggle edition

Turn a flat 2D image into a **walkable 3D world** exported as a single **GLB**, plus a JSON manifest,
a looping trailing-camera path for Remotion, and a three.js viewer with **Walk / Orbit / Cinematic**
modes (and a WebXR VR button). Runs headless on Kaggle with **no required input** — with no image it
builds a synthetic floor plan so the whole notebook finishes.

Two input paths, chosen automatically by looking at the image (or force one with `MODE`):

| Input looks like… | Path | Result |
|---|---|---|
| **A floor plan** — dark linework on a light background | **Plan** | discrete **rooms** + walls + door gaps + furniture |
| **Any other photo** — a room, a street, anything | **Photo** | a **room shell** (floor, walls, ceiling), the image as a depth-displaced backdrop and floor/ceiling bands, image-derived props, and a walkable nav grid |

**Plan path — how the detail is made**
1. Parse the plan → room polygons + a wall network (door gaps stay as real openings).
2. Extrude real geometry: per-room floor slabs, walls to height, **baseboards**, rugs, ceiling lights.
3. Apply **procedural textures** (wood, carpet, tile, plaster, fabric) with world-scale planar UVs.
4. Place **multi-part furniture** (a sofa is a base + back + arms + cushions + legs, not a box) with a
   deterministic settle + non-overlap solver.

**Photo path — how the world is made**
1. Estimate monocular depth (Depth-Anything V2 small via `transformers`; a classical fallback if no model).
2. Infer a room from the image: dimensions from the aspect/depth, floor/wall/ceiling tones sampled from the image.
3. Stand the image up as a **depth-displaced backdrop** on the far wall; continue it onto the floor and
   ceiling as thinner depth panels, so the source still reads from the spawn viewpoint while having real parallax.
4. Detect near blobs in the image → choose and place **multi-part props** with the same solver as the plan path.
5. Export a rasterised **navigation grid**, spawn point, zones, and a **subject + trailing-camera path**.

Both paths export one **GLB** with correct PBR materials/UVs, `scene_*.json` (spawn, zones, nav grid,
camera paths), `camera_path_*.json` (Remotion-ready), and `web/viewer_*.html`.

**Input:** drop your image into a Kaggle Dataset (it appears under `/kaggle/input/...`), or set `IMAGE_PATH`.
No image at all? It builds a synthetic floor plan so the whole notebook runs end-to-end.
**Output:** everything is written to `/kaggle/working/outputs/` (Kaggle collects `/kaggle/working` automatically).

> The photo path downloads a depth model from Hugging Face, so enable **Internet** in the Kaggle run
> settings; without it, the notebook falls back to a classical pseudo-depth and still builds a world.

> Coordinates in the GLB and manifests are **trimesh Z-up**. The viewer and the Remotion helper convert with
> `(x, y, z) → (x, z, -y)` (a −90° rotation about X) to get three.js Y-up.

## 📦 1 · Setup

In [ ]:
# Kaggle already ships numpy / opencv / pillow / matplotlib / pandas. Add trimesh + shapely.
import importlib, subprocess, sys
def _ensure(pairs):
    for mod, pkg in pairs:
        try:
            importlib.import_module(mod)
        except Exception:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
_ensure([("trimesh", "trimesh"), ("shapely", "shapely"),
         ("cv2", "opencv-python-headless"), ("PIL", "pillow"),
         ("mapbox_earcut", "mapbox-earcut"),          # trimesh needs an earcut engine
         ("transformers", "transformers")])           # photo path (uses Kaggle's torch)

import os, json, math, zipfile
from pathlib import Path
import numpy as np
import cv2
import trimesh
from trimesh.creation import extrude_polygon
from trimesh.visual.texture import TextureVisuals
from trimesh.visual.material import PBRMaterial
from shapely.geometry import Polygon
from shapely.ops import unary_union
import shapely
from PIL import Image
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import pandas as pd
print("trimesh", trimesh.__version__, "| shapely", shapely.__version__,
      "| cv2", cv2.__version__, "| numpy", np.__version__)
try:
    import torch
    HAS_TORCH = True
except Exception:
    HAS_TORCH = False
print("torch (for the photo depth model):", "available" if HAS_TORCH else "missing -> photo path falls back")


## 📥 2 · Choose the image (plan or photo)

In [ ]:
WORKDIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
OUT = WORKDIR / "outputs"
for _d in ["input", "3d", "textures", "visualizations", "web"]:
    (OUT / _d).mkdir(parents=True, exist_ok=True)
IN_EXT = {".png", ".jpg", ".jpeg", ".bmp", ".webp"}

# ---- how to feed your own image --------------------------------------------------
#   MODE = "auto"  -> decide by looking at the image (floor plan vs photo)
#   MODE = "plan"  -> force the plan path (rooms / walls / furniture)
#   MODE = "photo" -> force the photo path (depth relief mesh)
#   IMAGE_PATH     -> an explicit file (absolute, or a filename inside /kaggle/input)
MODE = "auto"
IMAGE_PATH = None

def generate_sample_plan(path):
    '''A clean synthetic apartment: white paper, black walls, a few door gaps.'''
    W, H, t = 1000, 720, 14
    img = np.full((H, W), 255, np.uint8)
    cv2.rectangle(img, (40, 40), (W - 40, H - 40), 0, t)      # outer shell
    cv2.line(img, (40, 360), (W - 40, 360), 0, t)             # horizontal divider
    cv2.line(img, (470, 40), (470, 360), 0, t)                # vertical (upper)
    cv2.line(img, (470, 360), (470, H - 40), 0, t)            # vertical (lower)
    cv2.line(img, (700, 360), (700, H - 40), 0, t)            # bathroom wall
    cv2.rectangle(img, (230, 352), (330, 368), 255, -1)       # door: living <-> kitchen
    cv2.rectangle(img, (462, 180), (478, 280), 255, -1)       # door: living <-> bedroom
    cv2.rectangle(img, (462, 500), (478, 560), 255, -1)       # door: kitchen <-> bath
    cv2.imwrite(str(path), img)
    return path

def looks_like_floorplan(img):
    '''Linework on a light background: low ink coverage and low colour saturation.'''
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, ink = cv2.threshold(g, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    frac = ink.mean() / 255.0
    sat = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)[..., 1].mean() / 255.0
    return (0.01 < frac < 0.40) and sat < 0.25

# pick exactly ONE image: explicit path, else the first image under /kaggle/input, else synthetic
chosen = None
if IMAGE_PATH:
    cand = Path(IMAGE_PATH)
    if not cand.exists() and Path("/kaggle/input").exists():
        hits = list(Path("/kaggle/input").rglob(cand.name))
        if hits:
            cand = hits[0]
    assert cand.exists(), f"IMAGE_PATH not found: {IMAGE_PATH}"
    chosen = cand
elif Path("/kaggle/input").exists():
    hits = sorted(p for p in Path("/kaggle/input").rglob("*")
                  if p.suffix.lower() in IN_EXT and not p.name.startswith("."))
    if hits:
        chosen = hits[0]
if chosen is None:
    chosen = OUT / "input" / "sample_apartment.png"
    generate_sample_plan(chosen)
    print("No image input -> using the built-in sample floor plan.")

img0 = cv2.imread(str(chosen))
assert img0 is not None, f"could not read image: {chosen}"
looks_plan = looks_like_floorplan(img0)
if MODE == "plan":
    USE_PLAN, USE_PHOTO = True, False
elif MODE == "photo":
    USE_PLAN, USE_PHOTO = False, True
else:
    USE_PLAN, USE_PHOTO = looks_plan, (not looks_plan)
print(f"image: {chosen}")
print(f"looks like a floor plan: {looks_plan}  |  MODE={MODE} -> plan={USE_PLAN} photo={USE_PHOTO}")
input_files = [chosen] if USE_PLAN else []


## 🔍 3 · Parse the plan into rooms + a wall network

In [ ]:
MIN_ROOM_PX = 2500          # ignore specks smaller than this
INK_K = 3                   # tiny close to tidy the scanned ink

def _interior_rooms(walls, H, W, k):
    '''Close door-sized gaps (kernel k) so rooms separate, then take interior free components.'''
    k = max(3, int(k) | 1)
    sep = cv2.morphologyEx(walls, cv2.MORPH_CLOSE, np.ones((k, k), np.uint8))
    free = cv2.bitwise_not(sep)
    ff = free.copy()
    cv2.floodFill(ff, np.zeros((H + 2, W + 2), np.uint8), (0, 0), 0)   # drop the outside
    n, labels, stats, cents = cv2.connectedComponentsWithStats(ff, connectivity=4)
    rooms = []
    for i in range(1, n):
        area = int(stats[i, cv2.CC_STAT_AREA])
        if area < MIN_ROOM_PX:
            continue
        comp = (labels == i).astype(np.uint8) * 255
        cnts, _ = cv2.findContours(comp, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        c = max(cnts, key=cv2.contourArea)
        poly = cv2.approxPolyDP(c, 0.012 * cv2.arcLength(c, True), True).reshape(-1, 2)
        x, y, w, h = cv2.boundingRect(c)
        rooms.append({"poly_px": poly.astype(float), "bbox_px": [int(x), int(y), int(w), int(h)],
                      "area_px": area, "centroid_px": [float(cents[i][0]), float(cents[i][1])]})
    return rooms

def segment_rooms(walls, H, W):
    '''Sweep the door-closing kernel; keep the smallest k that reaches the max room count.

    Door gaps make every room one connected free region; closing a door-sized kernel
    separates them, but too large a kernel starts eroding rooms. The true room count is
    the plateau maximum.'''
    hi = max(15, min(W, H) // 3)
    ks = sorted(set(list(range(9, hi, 15)) + [hi]))
    results = [(k, _interior_rooms(walls, H, W, k)) for k in ks]
    maxc = max(len(r) for _k, r in results)
    for k, rooms in results:
        if len(rooms) == maxc:
            return rooms, k
    return results[-1][1], results[-1][0]

def parse_floor_plan(path, meters_per_px=None):
    img = cv2.imread(str(path))
    if img is None:
        raise FileNotFoundError(path)
    H, W = img.shape[:2]
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, walls = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    walls = cv2.morphologyEx(walls, cv2.MORPH_CLOSE, np.ones((INK_K, INK_K), np.uint8))
    ink = int(np.count_nonzero(walls)) / float(W * H)
    if ink > 0.5:                                  # light background / dark lines? invert
        walls = cv2.bitwise_not(walls)
    rooms, k = segment_rooms(walls, H, W)
    rooms.sort(key=lambda r: -r["area_px"])
    for idx, r in enumerate(rooms):
        r["id"] = idx
    if meters_per_px is None:
        meters_per_px = 14.0 / max(W, H)           # assume the long side is ~14 m
    print(f"{Path(path).name}: {W}x{H}px -> {len(rooms)} rooms "
          f"(door-close k={k}) @ {meters_per_px*100:.2f} cm/px")
    return {"path": str(path), "W": W, "H": H, "mpp": meters_per_px,
            "rooms": rooms, "walls_mask": walls}

plans = [parse_floor_plan(p) for p in input_files] if USE_PLAN else []
if not USE_PLAN:
    print("(plan path skipped)")


## 🎨 4 · Procedural textures (no external assets)

In [ ]:
# ---------------------------------------------------------------- textures
def _save(name, arr):
    img = Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))
    img.save(OUT / "textures" / f"{name}.png")
    return img

def tex_wood(size=512, seed=3):
    rng = np.random.default_rng(seed)
    arr = np.zeros((size, size, 3), float)
    row = size // 6
    base = np.array([158, 112, 66], float)
    for i in range(6):
        col = base * (1.0 + (rng.random() - 0.5) * 0.28)
        arr[i*row:(i+1)*row] = col
        arr[i*row:i*row+2] = col * 0.45                 # seam
        g = np.sin(np.linspace(0, rng.uniform(4, 9) * math.pi, size))[None, :, None]
        arr[i*row:(i+1)*row] += (g * 10.0)
    arr += rng.normal(0, 5, arr.shape)
    return _save("wood", arr)

def tex_carpet(size=512, seed=7):
    rng = np.random.default_rng(seed)
    base = np.array([96, 92, 104], float)
    arr = np.tile(base, (size, size, 1))
    arr += rng.normal(0, 16, arr.shape)
    return _save("carpet", arr)

def tex_tiles(size=512, n=4, seed=11):
    rng = np.random.default_rng(seed)
    step, g = size // n, max(4, size // 90)
    base = np.array([214, 218, 222], float)
    arr = np.tile(base, (size, size, 1))
    for i in range(n):
        for j in range(n):
            arr[i*step+g:(i+1)*step, j*step+g:(j+1)*step] *= (1.0 + (rng.random() - 0.5) * 0.07)
    grout = np.array([150, 154, 160], float)
    for k in range(0, size, step):
        arr[k:k+g, :] = grout
        arr[:, k:k+g] = grout
    arr += rng.normal(0, 3, arr.shape)
    return _save("tile", arr)

def tex_plaster(size=512, seed=5):
    rng = np.random.default_rng(seed)
    base = np.array([232, 229, 223], float)
    arr = np.tile(base, (size, size, 1))
    arr += rng.normal(0, 4.0, (size, size, 1))
    # soften with a cheap box blur
    k = 3
    pad = np.pad(arr, ((k, k), (k, k), (0, 0)), mode="edge")
    acc = np.zeros_like(arr)
    for dx in range(-k, k + 1):
        for dy in range(-k, k + 1):
            acc += pad[k+dx:k+dx+size, k+dy:k+dy+size]
    arr = acc / ((2*k+1) ** 2)
    return _save("plaster", arr)

def tex_fabric(size=256, seed=13):
    rng = np.random.default_rng(seed)
    base = np.array([74, 78, 96], float)
    arr = np.tile(base, (size, size, 1))
    weave = (np.sin(np.arange(size) / 2.0)[:, None] + np.sin(np.arange(size) / 2.0)[None, :])
    arr += weave[:, :, None] * 6.0
    arr += rng.normal(0, 7, arr.shape)
    return _save("fabric", arr)

TEX = {"wood": tex_wood(), "carpet": tex_carpet(), "tile": tex_tiles(),
       "plaster": tex_plaster(), "fabric": tex_fabric()}
# world size (metres) that one texture tile covers, per material
TEX_WORLD = {"wood": 2.0, "carpet": 1.6, "tile": 1.0, "plaster": 2.4, "fabric": 0.6}

def build_materials():
    mats = {}
    rough = {"wood": 0.72, "carpet": 1.0, "tile": 0.35, "plaster": 0.92, "fabric": 0.95}
    for k, img in TEX.items():
        mats[k] = PBRMaterial(name=k, baseColorTexture=img,
                              baseColorFactor=[1.0, 1.0, 1.0, 1.0],
                              metallicFactor=0.0, roughnessFactor=rough[k])
    flat = {
        "baseboard": ([0.96, 0.95, 0.92], 0.6, 0.0),
        "wood_dark": ([0.30, 0.20, 0.12], 0.7, 0.0),
        "wood_mid":  ([0.54, 0.37, 0.22], 0.7, 0.0),
        "metal":     ([0.72, 0.73, 0.76], 0.35, 0.9),
        "white":     ([0.93, 0.94, 0.95], 0.45, 0.0),
        "counter":   ([0.20, 0.22, 0.25], 0.30, 0.1),
        "glass":     ([0.74, 0.86, 0.90], 0.10, 0.0),
        "fabric_lt": ([0.62, 0.64, 0.74], 0.95, 0.0),
        "rug":       ([0.52, 0.28, 0.30], 0.98, 0.0),
    }
    for k, (rgb, r, m) in flat.items():
        mats[k] = PBRMaterial(name=k, baseColorFactor=[rgb[0], rgb[1], rgb[2], 1.0],
                              roughnessFactor=r, metallicFactor=m)
    return mats

MATS = build_materials()

def apply_flat(mesh, name):
    mesh.visual = TextureVisuals(material=MATS[name])

def apply_tex(mesh, name, axis="xy"):
    v = mesh.vertices
    if axis == "xy":
        u, w = v[:, 0], v[:, 1]
    else:                                   # walls: wrap horizontally along the wall, up the height
        u, w = v[:, 0] + v[:, 1], v[:, 2]
    s = TEX_WORLD[name]
    mesh.visual = TextureVisuals(uv=np.column_stack([u / s, w / s]), material=MATS[name])

print("textures:", ", ".join(f"{k} {im.size[0]}px" for k, im in TEX.items()))
print("materials:", len(MATS))


## 🏗️ 5 · Extrude detailed geometry (floors, walls, baseboards)

In [ ]:
WALL_H = 2.8        # metres
FLOOR_T = 0.12      # floor slab thickness
BASE_H = 0.12       # baseboard height
BASE_W = 0.05       # baseboard thickness

ROOM_FLOOR = {"living_room": "wood", "bedroom": "carpet", "office": "wood",
              "kitchen": "tile", "bathroom": "tile"}

def poly_m(poly_px, H, mpp):
    pts = [(x * mpp, (H - y) * mpp) for x, y in poly_px]     # flip Y so the plan isn't mirrored
    p = Polygon(pts).buffer(0)
    return p if (not p.is_empty and p.area > 1e-6) else None

def ring(outer, width):
    inner = outer.buffer(-width)
    if inner.is_empty:
        return None
    r = outer.difference(inner)
    return r if (not r.is_empty and r.area > 1e-9) else None

def room_type(idx, total):
    if idx == 0:
        return "living_room"
    if total > 4 and idx == 1:
        return "bedroom"
    if total > 3 and idx == total - 1:
        return "kitchen"
    return ["bedroom", "office", "bathroom"][idx % 3]

def build_room_geometry(plan):
    H, mpp = plan["H"], plan["mpp"]
    rooms_m, meshes = [], []

    # --- floors (per room) + baseboards (per room) ---
    for idx, r in enumerate(plan["rooms"]):
        p = poly_m(r["poly_px"], H, mpp)
        if p is None:
            continue
        t = room_type(idx, len(plan["rooms"]))
        rooms_m.append({"id": r["id"], "type": t, "poly": p, "area": p.area, "bounds": list(p.bounds)})

        floor = extrude_polygon(p, FLOOR_T)
        floor.apply_translation([0, 0, -FLOOR_T])
        apply_tex(floor, ROOM_FLOOR[t], axis="xy")
        meshes.append(("floor", r["id"], floor))

        b = ring(p, BASE_W)
        if b is not None:
            bb = extrude_polygon(b, BASE_H)
            apply_flat(bb, "baseboard")
            meshes.append(("baseboard", r["id"], bb))

        if t in ("living_room", "bedroom"):        # an inset rug under the furniture
            rug = p.buffer(-0.8)
            if (not rug.is_empty) and rug.area > 1.0:
                rm = extrude_polygon(rug, 0.02)
                rm.apply_translation([0, 0, 0.004])
                apply_flat(rm, "rug")
                meshes.append(("rug", r["id"], rm))

        minx, miny, maxx, maxy = p.bounds         # a ceiling light fixture
        cl = trimesh.creation.cylinder(radius=0.16, height=0.06, sections=24)
        cl.apply_translation([(minx + maxx) / 2, (miny + maxy) / 2, WALL_H - 0.05])
        apply_flat(cl, "white")
        meshes.append(("light", r["id"], cl))

    # --- walls: extrude the wall-mask contours, keeping room interiors as real holes ---
    cnts, hier = cv2.findContours(plan["walls_mask"], cv2.RETR_CCOMP, cv2.CHAIN_APPROX_SIMPLE)
    hier = hier[0] if hier is not None else []
    n_wall = 0
    for i, c in enumerate(cnts):
        if cv2.contourArea(c) < (mpp ** 2) * 0.02:
            continue                                    # skip specks
        shell = poly_m(c.reshape(-1, 2), H, mpp)
        if shell is None:
            continue
        holes = []
        child = hier[i][2] if i < len(hier) else -1
        while child != -1:
            hp = poly_m(cnts[child].reshape(-1, 2), H, mpp)
            if hp is not None and hp.area > 1e-4:
                holes.append(list(hp.exterior.coords))
            child = hier[child][0]
        try:
            poly = Polygon(list(shell.exterior.coords), holes).buffer(0)
        except Exception:
            poly = shell
        if poly.is_empty or poly.area <= 1e-6:
            poly = shell
        try:
            wall = extrude_polygon(poly, WALL_H)
        except Exception:
            continue
        apply_tex(wall, "plaster", axis="xz")
        meshes.append(("wall", None, wall))
        n_wall += 1

    print(f"  {len(rooms_m)} rooms, {n_wall} wall pieces, "
          f"{sum(1 for k,_,_ in meshes if k=='baseboard')} baseboards")
    return rooms_m, meshes

scene_builds = []
for plan in plans:
    rooms_m, meshes = build_room_geometry(plan)
    scene_builds.append({"kind": "plan", "plan": plan, "rooms": rooms_m, "meshes": meshes})
if not scene_builds:
    print("(no plan scene built)")


## 🪑 6 · Multi-part furniture + settle/overlap solver

In [ ]:
def box_at(cx, cy, cz, w, d, h):
    m = trimesh.creation.box(extents=[w, d, h])
    m.apply_translation([cx, cy, cz])
    return m

# ---- part builders: author each item around (0,0) with the floor at z=0, +Y forward ----
def f_sofa(w, d, h):
    sh = h * 0.42
    p = [(box_at(0, 0, sh/2, w, d, sh), "fabric"),
         (box_at(0, d/2 - 0.11, sh + (h - sh)/2, w, 0.22, h - sh), "fabric"),
         (box_at(-w/2 + 0.10, 0, sh + 0.18, 0.20, d, 0.36), "fabric"),
         (box_at( w/2 - 0.10, 0, sh + 0.18, 0.20, d, 0.36), "fabric"),
         (box_at(0, -d*0.10, sh + 0.07, w - 0.5, d*0.55, 0.14), "fabric_lt")]
    for sx in (-1, 1):
        for sy in (-1, 1):
            p.append((box_at(sx*(w/2 - 0.10), sy*(d/2 - 0.10), 0.06, 0.10, 0.10, 0.12), "wood_dark"))
    return p

def f_armchair(w, d, h):
    return f_sofa(w, d, h)

def f_coffee_table(w, d, h):
    p = [(box_at(0, 0, h - 0.04, w, d, 0.06), "wood_mid")]
    for sx in (-1, 1):
        for sy in (-1, 1):
            p.append((box_at(sx*(w/2 - 0.06), sy*(d/2 - 0.06), (h - 0.04)/2, 0.07, 0.07, h - 0.04), "wood_dark"))
    return p

def f_tv_stand(w, d, h):
    p = [(box_at(0, 0, h/2, w, d, h), "wood_dark"),
         (box_at(0, 0, h + 0.35, w*0.9, 0.05, 0.7), "metal"),
         (box_at(0, 0.03, h + 0.35, w*0.82, 0.02, 0.6), "counter")]
    return p

def f_bed(w, d, h):
    p = [(box_at(0, 0, 0.16, w, d, 0.32), "wood_mid"),
         (box_at(0, 0.05, 0.40, w - 0.1, d - 0.1, 0.24), "white"),
         (box_at(0, d/2 - 0.02, 0.62, w, 0.06, 0.7), "wood_dark"),
         (box_at(-w*0.22, d*0.28, 0.54, w*0.4, 0.32, 0.14), "white"),
         (box_at( w*0.22, d*0.28, 0.54, w*0.4, 0.32, 0.14), "white"),
         (box_at(0, -d/2 + 0.15, 0.50, w - 0.15, 0.5, 0.08), "rug")]
    return p

def f_nightstand(w, d, h):
    return [(box_at(0, 0, h/2 + 0.08, w, d, h - 0.08), "wood_mid"),
            (box_at(0, -d/2 + 0.01, h*0.62, w - 0.06, 0.02, h*0.34), "wood_dark")] + \
           [(box_at(sx*(w/2 - 0.05), sy*(d/2 - 0.05), 0.04, 0.06, 0.06, 0.08), "wood_dark")
            for sx in (-1, 1) for sy in (-1, 1)]

def f_dresser(w, d, h):
    p = [(box_at(0, 0, h/2, w, d, h), "wood_mid")]
    for i in range(3):
        zc = h * (0.2 + i * 0.28)
        p.append((box_at(0, -d/2 - 0.005, zc, w - 0.08, 0.02, h * 0.2), "wood_dark"))
        p.append((box_at(0, -d/2 - 0.02, zc, w * 0.3, 0.03, 0.03), "metal"))
    return p

def f_wardrobe(w, d, h):
    p = [(box_at(0, 0, h/2, w, d, h), "wood_mid"),
         (box_at(0, -d/2 - 0.005, h*0.5, w/2 - 0.02, 0.02, h - 0.1), "wood_dark"),
         (box_at(0, -d/2 - 0.02, h*0.5, 0.03, 0.03, 0.18), "metal")]
    p += [(box_at(sx*(w/2 - 0.04), sy*(d/2 - 0.04), 0.06, 0.08, 0.08, 0.12), "wood_dark")
          for sx in (-1, 1) for sy in (-1, 1)]
    return p

def f_counter(w, d, h):
    return [(box_at(0, 0, (h - 0.04)/2, w, d, h - 0.04), "wood_mid"),
            (box_at(0, 0, h - 0.02, w + 0.03, d + 0.03, 0.05), "counter"),
            (box_at(-w*0.25, -d/2 - 0.005, h*0.5, w*0.3, 0.02, h*0.5), "wood_dark")]

def f_fridge(w, d, h):
    return [(box_at(0, 0, h/2, w, d, h), "white"),
            (box_at(0, -d/2 - 0.01, h*0.62, w - 0.06, 0.02, h*0.72), "metal"),
            (box_at(0, -d/2 - 0.03, h*0.75, 0.03, 0.05, 0.10), "metal")]

def f_table(w, d, h):
    p = [(box_at(0, 0, h, w, d, 0.05), "wood_mid")]
    p += [(box_at(sx*(w/2 - 0.07), sy*(d/2 - 0.07), (h - 0.05)/2, 0.07, 0.07, h - 0.05), "wood_dark")
          for sx in (-1, 1) for sy in (-1, 1)]
    return p

def f_tub(w, d, h):
    p = [(box_at(0, 0, h/2, w, d, h), "white")]
    inner = box_at(0, 0, h - 0.10, w - 0.16, d - 0.16, 0.20)
    p.append((inner, "counter"))     # dark recess reads as the basin
    return p

def f_sink_unit(w, d, h):
    return [(box_at(0, 0, (h - 0.04)/2, w, d, h - 0.04), "white"),
            (box_at(0, 0, h - 0.02, w + 0.03, d + 0.03, 0.05), "counter"),
            (box_at(0, 0, h + 0.02, w * 0.6, d * 0.6, 0.06), "white")]

def f_toilet(w, d, h):
    return [(box_at(0, -d/2 + 0.12, h*0.35, w, d*0.45, h*0.7), "white"),
            (box_at(0, d*0.15, 0.20, w*0.9, d*0.5, 0.4), "white"),
            (box_at(0, d*0.15, 0.42, w*0.75, d*0.42, 0.05), "white")]

def f_desk(w, d, h):
    return [(box_at(0, 0, h, w, d, 0.05), "wood_mid"),
            (box_at(0, d/2 - 0.03, h/2, w - 0.1, 0.04, h - 0.05), "wood_mid")] + \
           [(box_at(sx*(w/2 - 0.06), sy*(d/2 - 0.06), (h - 0.05)/2, 0.06, 0.06, h - 0.05), "metal")
            for sx in (-1, 1) for sy in (-1, 1)]

def f_chair(w, d, h):
    return [(box_at(0, 0, h*0.5, w, d, 0.06), "fabric_lt"),
            (box_at(0, d/2 - 0.04, h*0.5 + 0.22, w, 0.05, 0.44), "fabric_lt")] + \
           [(box_at(sx*(w/2 - 0.04), sy*(d/2 - 0.04), h*0.22, 0.05, 0.05, h*0.44), "metal")
            for sx in (-1, 1) for sy in (-1, 1)]

def f_shelf(w, d, h):
    p = [(box_at(-w/2 + 0.02, 0, h/2, 0.04, d, h), "wood_mid"),
         (box_at( w/2 - 0.02, 0, h/2, 0.04, d, h), "wood_mid"),
         (box_at(0, d/2 - 0.02, h/2, w, 0.03, h), "wood_dark")]
    for i in range(4):
        p.append((box_at(0, 0, h * (0.08 + i * 0.28), w, d, 0.03), "wood_mid"))
    return p

FURN = {
    "living_room": [("Sofa", (2.10, 0.85, 0.80), f_sofa), ("CoffeeTable", (1.10, 0.60, 0.45), f_coffee_table),
                    ("TvStand", (1.60, 0.45, 0.50), f_tv_stand), ("Armchair", (0.85, 0.85, 0.80), f_armchair)],
    "bedroom": [("Bed", (2.00, 1.60, 0.70), f_bed), ("Nightstand", (0.50, 0.45, 0.55), f_nightstand),
                ("Dresser", (1.20, 0.50, 0.90), f_dresser), ("Wardrobe", (1.20, 0.60, 2.00), f_wardrobe)],
    "kitchen": [("Counter", (2.40, 0.60, 0.90), f_counter), ("Counter2", (1.60, 0.60, 0.90), f_counter),
                ("Fridge", (0.70, 0.70, 1.80), f_fridge), ("Table", (1.10, 0.75, 0.75), f_table)],
    "bathroom": [("Tub", (1.70, 0.75, 0.55), f_tub), ("SinkUnit", (0.60, 0.50, 0.85), f_sink_unit),
                 ("Toilet", (0.40, 0.60, 0.75), f_toilet)],
    "office": [("Desk", (1.60, 0.80, 0.75), f_desk), ("Chair", (0.55, 0.55, 0.90), f_chair),
               ("Shelf", (1.00, 0.40, 1.90), f_shelf)],
}
MARGIN = 0.25

def place_furniture(rooms_m):
    items = []
    for r in rooms_m:
        minx, miny, maxx, maxy = r["bounds"]
        tpl = FURN.get(r["type"], FURN["bedroom"])
        cx, cy, row_h = minx + MARGIN, maxy - MARGIN, 0.0
        for name, (w, d, h), _fn in tpl:
            if cx + w > maxx - MARGIN:
                cx, cy, row_h = minx + MARGIN, cy - row_h - MARGIN, 0.0
            if cy - d < miny:
                break
            items.append({"type": name, "room_id": r["id"], "size": [w, d, h],
                          "position": [cx, cy - d]})
            cx += w + MARGIN
            row_h = max(row_h, d)
    return items

def solve_placement(items, rooms_m, iters=80):
    by_id = {r["id"]: r for r in rooms_m}
    for _ in range(iters):
        moved = False
        for a in items:                                   # keep inside the room bounds
            r = by_id[a["room_id"]]
            minx, miny, maxx, maxy = r["bounds"]
            w, d, _ = a["size"]
            nx = min(max(a["position"][0], minx + 0.02), maxx - w - 0.02)
            ny = min(max(a["position"][1], miny + 0.02), maxy - d - 0.02)
            if abs(nx - a["position"][0]) > 1e-6 or abs(ny - a["position"][1]) > 1e-6:
                a["position"][0], a["position"][1] = nx, ny
                moved = True
        for i in range(len(items)):                       # separate overlaps within a room
            for j in range(i + 1, len(items)):
                a, b = items[i], items[j]
                if a["room_id"] != b["room_id"]:
                    continue
                ax, ay = a["position"]; bx, by = b["position"]
                aw, ad = a["size"][0], a["size"][1]
                bw, bd = b["size"][0], b["size"][1]
                ox = min(ax + aw, bx + bw) - max(ax, bx)
                oy = min(ay + ad, by + bd) - max(ay, by)
                if ox > 0 and oy > 0:
                    if ox < oy:
                        s = -1 if ax < bx else 1
                        a["position"][0] += s * (ox/2 + 0.01); b["position"][0] -= s * (ox/2 + 0.01)
                    else:
                        s = -1 if ay < by else 1
                        a["position"][1] += s * (oy/2 + 0.01); b["position"][1] -= s * (oy/2 + 0.01)
                    moved = True
        if not moved:
            break
    return items

def furniture_meshes(items):
    out = []
    for it in items:
        w, d, h = it["size"]
        cx, cy = it["position"][0] + w/2, it["position"][1] + d/2
        fn = it["_fn"]
        for mesh, matname in fn(w, d, h):
            mesh.apply_translation([cx, cy, 0.0])
            if matname in TEX:
                apply_tex(mesh, matname, axis="xy")
            else:
                apply_flat(mesh, matname)
            out.append((it, mesh))
    return out

for sb in scene_builds:
    items = solve_placement(place_furniture(sb["rooms"]), sb["rooms"])
    tmap = {r["id"]: r["type"] for r in sb["rooms"]}
    fbn = {name: fn for t in FURN.values() for name, _s, fn in t}
    for it in items:
        it["_room_type"] = tmap[it["room_id"]]
        it["_fn"] = fbn[it["type"]]
    parts = furniture_meshes(items)
    sb["furniture"] = items
    sb["meshes"] += [("furniture", it["room_id"], m) for it, m in parts]
    print(f"  scene: {len(sb['rooms'])} rooms, {len(items)} furniture items, "
          f"{len(parts)} furniture parts")


## 🌆 6b · Photo path — monocular depth → an explorable world (not just a relief)

The image becomes one surface *inside* a walkable room: a depth-displaced backdrop on the far wall,
depth bands on the floor/ceiling, sampled materials on the shell, and image-derived props. Everything
gets a nav grid, spawn, zones, and camera paths for exploration or animation.

In [ ]:
# Runs only when USE_PHOTO is true (the image is not a floor plan).
# The result is not a flat relief: we build a walkable room around the image,
# use its depth as a displaced backdrop, and furnish it with collision-ready props.
_MODEL_CACHE = {}

def estimate_depth(img_bgr):
    """Monocular depth. Tries Depth-Anything V2 (small) via transformers; falls back to a
    classical pseudo-depth when torch/transformers are unavailable (no internet, no GPU)."""
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    if HAS_TORCH:
        try:
            from transformers import pipeline
            from PIL import Image as _Image
            pipe = _MODEL_CACHE.get("depth")
            if pipe is None:
                pipe = pipeline("depth-estimation", model="depth-anything/Depth-Anything-V2-Small-hf")
                _MODEL_CACHE["depth"] = pipe
            out = pipe(_Image.fromarray(rgb))
            d = np.asarray(out["depth"], dtype=np.float32)
            d = (d - d.min()) / (float(np.ptp(d)) + 1e-6)      # 0..1, 1 = nearest
            return d, "depth-anything-v2-small"
        except Exception as e:
            print("  depth model unavailable:", type(e).__name__, "-> classical pseudo-depth")
    else:
        print("  torch not available -> classical pseudo-depth "
              "(enable Internet and install torch for a real depth model)")
    g = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY).astype(np.float32) / 255.0
    g = cv2.GaussianBlur(g, (0, 0), 3.0)
    d = 1.0 - g                                                # darker = nearer (a rough guess)
    d = (d - d.min()) / (float(np.ptp(d)) + 1e-6)
    return d, "pseudo-depth (luminance)"


def _panel_mesh(rgb, depth, width, height, plane="vertical", max_side=256, relief=0.22,
                name="photo_backdrop"):
    """A textured, depth-displaced image panel.
    vertical   : spans X x Z, depth pushes along Y (into the room when push=-1).
    horizontal : spans X x Y, depth pushes along Z (down when push=-1).
    The caller places the returned mesh by translation."""
    H0, W0 = depth.shape[:2]
    s = min(1.0, max_side / max(H0, W0))
    W, H = max(8, int(round(W0 * s))), max(8, int(round(H0 * s)))
    img = cv2.resize(rgb, (W, H), interpolation=cv2.INTER_AREA)
    d = cv2.resize(depth, (W, H), interpolation=cv2.INTER_LINEAR)
    d = cv2.GaussianBlur(d, (0, 0), 1.0)
    xs = np.linspace(-width / 2, width / 2, W)
    vs = np.linspace(-height / 2, height / 2, H)
    X, V = np.meshgrid(xs, vs)
    disp = (d - 0.5) * relief
    if plane == "vertical":
        verts = np.column_stack([X.ravel(), disp.ravel(), V.ravel()])
    else:
        verts = np.column_stack([X.ravel(), V.ravel(), disp.ravel()])
    idx = np.arange(H * W).reshape(H, W)
    a, b = idx[:-1, :-1].ravel(), idx[:-1, 1:].ravel()
    c, e = idx[1:, :-1].ravel(), idx[1:, 1:].ravel()
    faces = np.vstack([np.column_stack([a, b, e]), np.column_stack([a, e, c])])
    U, VV = np.meshgrid(np.linspace(0, 1, W), np.linspace(0, 1, H))
    mesh = trimesh.Trimesh(vertices=verts, faces=faces, process=False)
    mat = PBRMaterial(name=name, baseColorTexture=Image.fromarray(img),
                      baseColorFactor=[1.0, 1.0, 1.0, 1.0],
                      metallicFactor=0.0, roughnessFactor=0.92)
    mesh.visual = TextureVisuals(uv=np.column_stack([U.ravel(), VV.ravel()]), material=mat)
    return mesh


def _band_color(img_bgr, y0, y1):
    """Median RGB of a horizontal band of the source image."""
    h, w = img_bgr.shape[:2]
    crop = img_bgr[int(h * y0):max(int(h * y0) + 1, int(h * y1))]
    small = cv2.resize(crop, (32, 32), interpolation=cv2.INTER_AREA)
    return (np.median(small.reshape(-1, 3), axis=0)[::-1] / 255.0).tolist()


def _photo_prop_seeds(img_bgr, depth, k=5):
    """Find near / distinct blobs in the image: cheap stand-ins for semantic objects."""
    h, w = depth.shape[:2]
    d = cv2.resize(depth, (96, 96), interpolation=cv2.INTER_AREA)
    near = (d > np.percentile(d, 62)).astype(np.uint8) * 255
    near[:int(96 * 0.22), :] = 0        # ignore ceiling band
    near[int(96 * 0.94):, :] = 0
    near = cv2.morphologyEx(near, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    near = cv2.morphologyEx(near, cv2.MORPH_CLOSE, np.ones((7, 7), np.uint8))
    n, lab, stats, cents = cv2.connectedComponentsWithStats(near, connectivity=8)
    seeds = []
    for i in range(1, n):
        area = int(stats[i, cv2.CC_STAT_AREA])
        if area < 70:
            continue
        u, v = float(cents[i][0]) / 96.0, float(cents[i][1]) / 96.0
        seeds.append({"u": u, "v": v, "area": area,
                      "bright": float(np.mean(cv2.resize(
                          cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY), (96, 96))[
                              int(v * 95), int(u * 95)])) / 255.0})
    seeds.sort(key=lambda s: -s["area"])
    return seeds[:k]


def build_photo_world(img_bgr, depth, method):
    """Turn one image + depth into an explorable room: shell, displaced photo walls,
    ceiling lights, image-derived props, spawn point and zones."""
    H0, W0 = img_bgr.shape[:2]
    room_w = float(np.clip(8.0 * (W0 / max(H0, 1)) ** 0.15, 6.5, 10.5))
    room_d = float(np.clip(6.5 + 2.0 * (0.5 - float(np.median(depth))), 5.5, 8.5))
    room_h = WALL_H
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    meshes = []

    def flat_box(extents, center, name, rgb_factor, rough=0.9):
        m = trimesh.creation.box(extents=extents)
        m.apply_translation(center)
        m.visual = TextureVisuals(material=PBRMaterial(
            name=name, baseColorFactor=[float(c) for c in rgb_factor] + [1.0],
            metallicFactor=0.0, roughnessFactor=rough))
        meshes.append((name, None, m))
        return m

    # --- walkable shell -------------------------------------------------------
    floor_rgb = _band_color(img_bgr, 0.72, 1.0)
    wall_rgb = _band_color(img_bgr, 0.25, 0.65)
    ceil_rgb = _band_color(img_bgr, 0.0, 0.20)
    flat_box([room_w, room_d, 0.12], [0, 0, -0.06], "photo_floor",
             [min(1, c * 1.05) for c in floor_rgb], 0.82)
    flat_box([room_w, room_d, 0.10], [0, 0, room_h + 0.05], "photo_ceiling",
             [min(1, c * 1.15 + 0.12) for c in ceil_rgb], 0.95)
    t = 0.12
    flat_box([room_w, t, room_h], [0, room_d / 2 - t / 2, room_h / 2],
             "wall_back", [min(1, c * 0.85 + 0.15) for c in wall_rgb])
    flat_box([room_w, t, room_h], [0, -room_d / 2 + t / 2, room_h / 2],
             "wall_front", [min(1, c * 0.85 + 0.15) for c in wall_rgb])
    flat_box([t, room_d, room_h], [-room_w / 2 + t / 2, 0, room_h / 2],
             "wall_left", [min(1, c * 0.85 + 0.15) for c in wall_rgb])
    flat_box([t, room_d, room_h], [room_w / 2 - t / 2, 0, room_h / 2],
             "wall_right", [min(1, c * 0.85 + 0.15) for c in wall_rgb])

    # --- the image itself, now geometry you can walk up to --------------------
    back = _panel_mesh(rgb, depth, room_w * 0.84, room_h * 0.88, "vertical",
                       name="photo_backdrop")
    back.apply_translation([0, room_d / 2 - 0.26, room_h / 2 + 0.02])
    meshes.append(("photo_backdrop", None, back))

    floor_band = rgb[int(H0 * 0.70):, :, :]
    depth_band = depth[int(H0 * 0.70):, :]
    fband = _panel_mesh(floor_band, depth_band, room_w * 0.84, 2.0, "horizontal",
                        relief=0.10, name="photo_floor_band")
    fband.apply_translation([0, room_d / 2 - 1.35, 0.075])
    meshes.append(("photo_floor_band", None, fband))

    ceil_band = rgb[:max(1, int(H0 * 0.18)), :, :]
    depth_ceil = depth[:max(1, int(H0 * 0.18)), :]
    cband = _panel_mesh(ceil_band, depth_ceil, room_w * 0.84, 1.6, "horizontal",
                        relief=0.08, name="photo_ceiling_band")
    cband.apply_translation([0, room_d / 2 - 1.15, room_h - 0.07])
    meshes.append(("photo_ceiling_band", None, cband))

    for i, lx in enumerate(np.linspace(-room_w / 3, room_w / 3, 3)):
        lamp = trimesh.creation.cylinder(radius=0.16, height=0.06, sections=24)
        lamp.apply_translation([float(lx), 0.0, room_h - 0.10])
        apply_flat(lamp, "white")
        meshes.append(("light", i, lamp))

    # --- props: image blobs become furniture, then the existing solver places it
    fbn = {name: fn for t in FURN.values() for name, _s, fn in t}
    sizes = {name: size for t in FURN.values() for name, size, _fn in t}
    seeds = _photo_prop_seeds(img_bgr, depth)
    prop_names = []
    for s in seeds:
        if s["bright"] > 0.72 and s["area"] < 300:
            name = "Shelf"
        elif s["area"] > 1400:
            name = "Sofa"
        elif s["bright"] < 0.35:
            name = "TvStand" if s["u"] > 0.5 else "Dresser"
        else:
            name = ["CoffeeTable", "Armchair", "Desk"][len(prop_names) % 3]
        prop_names.append(name)
        if len(prop_names) >= 5:
            break
    for name in ["Sofa", "CoffeeTable", "TvStand", "Armchair"]:
        if len(prop_names) >= 4:
            break
        prop_names.append(name)

    spawn = [0.0, -room_d / 2 + 1.0, 1.6]           # trimesh: X, Y, Z-up
    bounds = [-room_w / 2 + 0.35, -room_d / 2 + 0.9,
              room_w / 2 - 0.35, room_d / 2 - 0.7]
    pseudo_room = {"id": 0, "type": "living_room", "bounds": bounds}
    items = []
    for i, name in enumerate(prop_names):
        w, d, h = sizes.get(name, (1.0, 0.8, 0.8))
        seed = seeds[i] if i < len(seeds) else {"u": 0.5, "v": 0.6}
        px = (seed["u"] - 0.5) * room_w * 0.72
        py = room_d / 2 - 1.1 - seed["v"] * (room_d * 0.5)
        items.append({"type": name, "room_id": 0, "size": [w, d, h],
                      "position": [float(px - w / 2), float(py - d / 2)],
                      "_fn": fbn[name], "_room_type": "living_room"})
    items = solve_placement(items, [pseudo_room])
    for it in items:                                  # keep the spawn circle clear
        x, y = it["position"]
        w, d, _ = it["size"]
        if x - 0.45 < spawn[0] < x + w + 0.45 and y - 0.45 < spawn[1] < y + d + 0.45:
            maxy = bounds[3]
            it["position"][1] = min(maxy - d, spawn[1] + 1.3 + d)
    items = solve_placement(items, [pseudo_room])
    for it in items:
        for mesh, matname in it["_fn"](*it["size"]):
            x, y = it["position"]
            mesh.apply_translation([x + it["size"][0] / 2, y + it["size"][1] / 2, 0])
            if matname in TEX:
                apply_tex(mesh, matname, axis="xy")
            else:
                apply_flat(mesh, matname)
            meshes.append(("furniture", 0, mesh))

    zones = [
        {"name": "entry", "position": [0.0, -room_d / 2 + 1.2, 1.6]},
        {"name": "centre", "position": [0.0, 0.0, 1.6]},
        {"name": "feature wall", "position": [0.0, room_d / 2 - 2.0, 1.6]},
    ]
    print(f"  photo world: {room_w:.1f} x {room_d:.1f} x {room_h:.1f} m, "
          f"{len(items)} props, {len(meshes)} meshes ({method})")
    return {
        "kind": "photo", "plan": {"path": str(chosen), "mpp": 0.0,
                                  "room": {"w": room_w, "d": room_d, "h": room_h}},
        "rooms": [], "furniture": items, "meshes": meshes,
        "spawn": {"position": spawn, "yaw": 0.0}, "zones": zones,
        "walk_bounds": bounds, "depth_method": method,
        "depth": depth, "axis": "z-up",
    }

if USE_PHOTO:
    depth, method = estimate_depth(img0)
    cv2.imwrite(str(OUT / "visualizations" / "depth.png"), (depth * 255).astype(np.uint8))
    scene_builds.append(build_photo_world(img0, depth, method))
    print(f"  photo path: world built from {Path(chosen).name}")
else:
    print("(photo path skipped)")

## 🎮 7 · Export GLB / GLTF + JSON manifest

In [ ]:
import heapq

def nav_grid(sb, res=168):
    """Rasterised walkability in trimesh XY (Z is up). 1 = blocked, 0 = walkable."""
    if sb.get("walk_bounds"):
        minx, miny, maxx, maxy = [float(v) for v in sb["walk_bounds"]]
    elif sb["rooms"]:
        minx = min(r["bounds"][0] for r in sb["rooms"])
        miny = min(r["bounds"][1] for r in sb["rooms"])
        maxx = max(r["bounds"][2] for r in sb["rooms"])
        maxy = max(r["bounds"][3] for r in sb["rooms"])
    else:
        minx, miny, maxx, maxy = -5.0, -5.0, 5.0, 5.0
    minx -= 0.35; miny -= 0.35; maxx += 0.35; maxy += 0.35
    cell = (maxx - minx) / float(res - 1)
    plan = sb.get("plan", {})
    mask = plan.get("walls_mask")
    mpp = float(plan.get("mpp") or 0.0)
    H = int(plan.get("H") or 0)
    feet = [(it["position"][0], it["position"][1], it["size"][0], it["size"][1])
            for it in sb["furniture"]]
    rows = []
    for j in range(res):
        y = miny + j * cell
        line = []
        for i in range(res):
            x = minx + i * cell
            blocked = False
            if mask is not None and mpp > 0:
                px, py = int(round(x / mpp)), int(round(H - y / mpp))
                if not (0 <= px < mask.shape[1] and 0 <= py < mask.shape[0]) or mask[py, px] > 0:
                    blocked = True
            elif sb.get("walk_bounds"):
                wx0, wy0, wx1, wy1 = sb["walk_bounds"]
                if not (wx0 <= x <= wx1 and wy0 <= y <= wy1):
                    blocked = True
            if not blocked:
                for fx, fy, fw, fd in feet:
                    if fx - 0.12 <= x <= fx + fw + 0.12 and fy - 0.12 <= y <= fy + fd + 0.12:
                        blocked = True
                        break
            line.append("1" if blocked else "0")
        rows.append("".join(line))
    return {"min": [round(minx, 4), round(miny, 4)], "cell": round(cell, 5),
            "rows": res, "cols": res, "blocked": rows}


def free_spot(nav, wanted):
    """Nearest walkable point to a desired spawn, with a little clearance."""
    minx, miny = nav["min"]
    cell = nav["cell"]
    grid = nav["blocked"]
    cols, rows = nav["cols"], nav["rows"]

    def walkable(i, j):
        return (0 <= i < cols and 0 <= j < rows and grid[j][i] == "0")

    ci = int(round((wanted[0] - minx) / cell))
    cj = int(round((wanted[1] - miny) / cell))
    best = None
    for r in range(0, 22):
        for dj in range(-r, r + 1):
            for di in range(-r, r + 1):
                i, j = ci + di, cj + dj
                if not walkable(i, j):
                    continue
                if not all(walkable(i + a, j + b) for a, b in
                           [(-2, 0), (2, 0), (0, -2), (0, 2), (2, 2), (-2, -2)]):
                    continue
                p = [minx + i * cell, miny + j * cell]
                dist = (p[0] - wanted[0]) ** 2 + (p[1] - wanted[1]) ** 2
                if best is None or dist < best[0]:
                    best = (dist, p)
        if best is not None:
            break
    return best[1] if best else [float(wanted[0]), float(wanted[1])]


def _nav_cell(nav, x, y):
    return (int(round((x - nav["min"][0]) / nav["cell"])),
            int(round((y - nav["min"][1]) / nav["cell"])))


def _nav_free(nav, i, j):
    return (0 <= i < nav["cols"] and 0 <= j < nav["rows"] and nav["blocked"][j][i] == "0")


def _segment_free(nav, a, b):
    n = max(2, int(math.hypot(b[0]-a[0], b[1]-a[1]) / (nav["cell"] * 0.6)))
    for t in np.linspace(0, 1, n):
        i, j = _nav_cell(nav, a[0] + (b[0]-a[0]) * t, a[1] + (b[1]-a[1]) * t)
        if not _nav_free(nav, i, j):
            return False
    return True


def _astar(nav, a, b):
    """Route between two points on the walkability grid (8-neighbour A*)."""
    start, goal = _nav_cell(nav, *a), _nav_cell(nav, *b)
    if start == goal:
        return [list(a), list(b)]
    heap = [(0.0, start)]
    came, cost = {}, {start: 0.0}
    dirs = [(1,0),(-1,0),(0,1),(0,-1),(1,1),(1,-1),(-1,1),(-1,-1)]
    while heap:
        _, cur = heapq.heappop(heap)
        if cur == goal:
            break
        for di, dj in dirs:
            nxt = (cur[0]+di, cur[1]+dj)
            if not _nav_free(nav, *nxt):
                continue
            if di and dj and not (_nav_free(nav, cur[0]+di, cur[1]) and
                                  _nav_free(nav, cur[0], cur[1]+dj)):
                continue                                   # no cutting corners
            step = math.hypot(di, dj)
            g = cost[cur] + step
            if g < cost.get(nxt, 1e18):
                cost[nxt], came[nxt] = g, cur
                heapq.heappush(heap, (g + math.hypot(goal[0]-nxt[0], goal[1]-nxt[1]), nxt))
    if goal not in came:
        return [list(a), list(b)]
    chain, cur = [goal], goal
    while cur != start:
        cur = came[cur]
        chain.append(cur)
    chain.reverse()
    pts = [[nav["min"][0] + i*nav["cell"], nav["min"][1] + j*nav["cell"]] for i, j in chain]
    return [list(a)] + pts[4:-4:4] + [list(b)]


def _simplify(nav, pts):
    """String-pull the A* route: keep only corners that are actually needed."""
    out = [pts[0]]
    i = 0
    while i < len(pts) - 1:
        j = len(pts) - 1
        while j > i + 1 and not _segment_free(nav, pts[i], pts[j]):
            j -= 1
        out.append(pts[j])
        i = j
    return out


def _catmull(p0, p1, p2, p3, t):
    return 0.5 * ((2 * p1) + (-p0 + p2) * t +
                  (2 * p0 - 5 * p1 + 4 * p2 - p3) * t * t +
                  (-p0 + 3 * p1 - 3 * p2 + p3) * t * t * t)


def build_camera_paths(sb, spawn_xy, nav=None, frames=360, per_leg=8):
    """A looping subject path plus a camera that trails it. Coordinates are trimesh
    (X, Y, Z-up); the viewer / Remotion convert with (x, y, z) -> (x, z, -y)."""
    if sb["rooms"]:
        way = sorted(sb["rooms"], key=lambda r: -r["area"])
        pts = [[(r["bounds"][0] + r["bounds"][2]) / 2,
                (r["bounds"][1] + r["bounds"][3]) / 2] for r in way[:6]]
    else:
        pts = [[z["position"][0], z["position"][1]] for z in sb["zones"]]
        pts += [[-2.2, 0.0], [2.2, 0.0]]
    if len(pts) < 3:
        cx, cy = spawn_xy
        r = 2.0
        pts = [[cx + r * math.cos(a), cy + r * math.sin(a)]
               for a in np.linspace(0, 2 * math.pi, 8, endpoint=False)]
    if nav is not None:                       # route through doorways, never through walls
        safe = [free_spot(nav, p) for p in pts]
        routed = []
        for k in range(len(safe)):
            seg = _astar(nav, safe[k], safe[(k + 1) % len(safe)])
            routed += _simplify(nav, seg)[:-1]
        if len(routed) >= 3:
            pts = routed
    loop = []
    n = len(pts)
    for i in range(n):
        p0, p1 = np.array(pts[(i - 1) % n], float), np.array(pts[i], float)
        p2, p3 = np.array(pts[(i + 1) % n], float), np.array(pts[(i + 2) % n], float)
        for s in range(per_leg):
            t = s / float(per_leg)
            loop.append([float(_catmull(p0[k], p1[k], p2[k], p3[k], t)) for k in (0, 1)])
    m = len(loop)
    subject, camera = [], []
    for i, (x, y) in enumerate(loop):
        nxt = np.array(loop[(i + 1) % m], float)
        prv = np.array(loop[(i - 1) % m], float)
        tan = nxt - prv
        ln = float(np.linalg.norm(tan)) + 1e-6
        tan /= ln
        lag = loop[(i - max(1, m // 40)) % m]
        sx, sy = x, y
        if nav is not None and not _nav_free(nav, *_nav_cell(nav, sx, sy)):
            sx, sy = free_spot(nav, (sx, sy))    # smooth curve may clip a corner
        subject.append({"frame": int(round(i * frames / m)) % frames,
                        "position": [round(sx, 3), round(sy, 3), 0.35],
                        "look_at": [round(x + tan[0], 3), round(y + tan[1], 3), 0.6]})
        cx, cy = sx - tan[0] * 2.3 - tan[1] * 0.6, sy - tan[1] * 2.3 + tan[0] * 0.6
        if nav is not None and not _nav_free(nav, *_nav_cell(nav, cx, cy)):
            for back, side in [(1.7, 0.4), (1.1, 0.3), (0.5, 0.0), (0.0, 0.0)]:
                tx = sx - tan[0] * back - tan[1] * side
                ty = sy - tan[1] * back + tan[0] * side
                if _nav_free(nav, *_nav_cell(nav, tx, ty)):
                    cx, cy = tx, ty
                    break
        camera.append({"frame": int(round(i * frames / m)) % frames,
                       "position": [round(cx, 3), round(cy, 3), 1.65],
                       "look_at": [round(lag[0], 3), round(lag[1], 3), 0.95]})
    return {"subject": subject, "camera": camera, "frames": frames}


def export_scene(sb, scene_id):
    scene = trimesh.Scene()
    for kind, room_id, m in sb["meshes"]:
        base = f"{kind}_{room_id}" if room_id is not None else kind
        scene.add_geometry(m, node_name=f"{base}_{len(scene.geometry)}")
    scene.metadata["axis"] = "z-up"          # glTF is Y-up; viewer/Remotion rotate -90deg X
    glb = OUT / "3d" / f"scene_{scene_id}.glb"
    scene.export(str(glb))
    try:
        gdir = OUT / "3d" / "gltf"               # keep the multi-buffer glTF tidy in a subdir
        gdir.mkdir(parents=True, exist_ok=True)
        scene.export(str(gdir / f"scene_{scene_id}.gltf"))
    except Exception as e:
        print("  (gltf export skipped:", e, ")")

    nav = nav_grid(sb)
    wanted = sb.get("spawn", {}).get("position")
    if wanted is None and sb["rooms"]:
        r = max(sb["rooms"], key=lambda r: r["area"])
        wanted = [(r["bounds"][0] + r["bounds"][2]) / 2,
                  (r["bounds"][1] + r["bounds"][3]) / 2]
    elif wanted is None:
        wanted = [0.0, 0.0]
    sx, sy = free_spot(nav, wanted[:2])
    spawn = {"position": [round(sx, 3), round(sy, 3), 1.65], "yaw": float(sb.get("spawn", {}).get("yaw", 0.0))}

    if sb.get("zones"):
        zones = [{"name": z["name"], "position": [round(v, 3) for v in z["position"]]}
                 for z in sb["zones"]]
    else:
        zones = [{"name": r["type"] + "_" + str(r["id"]),
                  "position": [round((r["bounds"][0] + r["bounds"][2]) / 2, 3),
                               round((r["bounds"][1] + r["bounds"][3]) / 2, 3), 1.65]}
                 for r in sb["rooms"]]

    paths = build_camera_paths(sb, (sx, sy), nav=nav)
    bounds = scene.bounds.tolist() if scene.bounds is not None else [[0, 0, 0], [0, 0, 0]]
    meta = {
        "scene_id": scene_id,
        "kind": sb.get("kind", "plan"),
        "source": Path(sb["plan"]["path"]).name,
        "scale_m_per_px": sb["plan"]["mpp"],
        "axis": "z-up",
        "axis_note": "trimesh coordinates; convert to three.js with (x, y, z) -> (x, z, -y)",
        "bounds_m": bounds,
        "textures": sorted(TEX.keys()) if sb.get("kind", "plan") == "plan" else ["photo (embedded)"],
        "depth_method": sb.get("depth_method"),
        "detail": {"wall_height_m": WALL_H, "floor_thickness_m": FLOOR_T,
                   "baseboards": True, "multi_part_furniture": True,
                   "walkable": True, "explorable": True},
        "rooms": [{"id": r["id"], "type": r["type"], "area_m2": round(r["area"], 2),
                   "bounds": [round(x, 3) for x in r["bounds"]]} for r in sb["rooms"]],
        "furniture": [{"type": it["type"], "room_id": it["room_id"], "size": it["size"],
                       "position": [round(v, 3) for v in it["position"]]} for it in sb["furniture"]],
        "spawn": spawn,
        "zones": zones,
        "nav_grid": nav,
        "interaction_points": [{"id": "zone_" + z["name"].replace(" ", "_"), "type": "zone",
                                "position": z["position"], "actions": ["enter", "teleport"]}
                               for z in zones],
        "camera_paths": {"subject": paths["subject"], "camera": paths["camera"]},
        "camera_settings": {"fps": 30, "frames": paths["frames"], "loop": True,
                            "interpolation": "catmull-rom",
                            "trailing": {"position_lag_m": 2.3, "look_at_lag_frames": 9,
                                         "smoothing": 0.14}},
        "lighting": {"ambient": 0.6, "directional": 1.4, "shadows": True},
        "physics": {"gravity": [0, 0, -9.81], "collision": "nav-grid + aabb",
                    "solver": "settle-and-separate", "iterations": 80,
                    "eye_height_m": 1.65, "radius_m": 0.28},
    }
    (OUT / "3d" / f"scene_{scene_id}.json").write_text(json.dumps(meta, indent=2))
    path_file = OUT / "3d" / f"camera_path_{scene_id}.json"
    path_file.write_text(json.dumps({
        "version": 1, "axis": "z-up", "fps": 30, "frames": paths["frames"], "loop": True,
        "interpolation": "catmull-rom", "settings": meta["camera_settings"]["trailing"],
        "subject_path": paths["subject"], "camera_path": paths["camera"]}, indent=2))
    return glb, meta

exports = [export_scene(sb, i + 1) for i, sb in enumerate(scene_builds)]
for glb, meta in exports:
    print(f"  exported {glb.name}  ({glb.stat().st_size/1024:.0f} KB)  "
          f"{len(meta['rooms'])} rooms, {len(meta['furniture'])} items, "
          f"{len(meta['camera_paths']['camera'])} camera keys")

## 🌐 8 · Web viewer (three.js ESM + GLTFLoader + OrbitControls)

In [ ]:
VIEWER = r"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8"/>
<meta name="viewport" content="width=device-width, initial-scale=1"/>
<title>__TITLE__</title>
<style>
  html,body{margin:0;height:100%;overflow:hidden;background:#0e1116;color:#e8eef7;
            font-family:system-ui,Arial,sans-serif}
  #hud{position:fixed;top:12px;left:12px;z-index:5;background:rgba(14,17,22,.82);
       padding:10px 14px;border-radius:10px;font-size:13px;line-height:1.55;max-width:340px}
  #hud b{color:#7fd1ff}
  .modes{display:flex;gap:6px;margin:8px 0 4px}
  button.mode{background:#223043;color:#d7e8ff;border:1px solid #3c587a;border-radius:6px;
              padding:5px 9px;cursor:pointer;font-size:12px}
  button.mode.active{background:#2f7fb8;border-color:#7fd1ff;color:#fff}
  #hint{color:#9bb0c8;font-size:12px}
  #cross{position:fixed;left:50%;top:50%;width:6px;height:6px;margin:-3px;border-radius:50%;
         background:#7fd1ff;opacity:.8;z-index:4;display:none}
  #label{position:fixed;bottom:12px;left:12px;z-index:5;background:rgba(14,17,22,.7);
         padding:6px 10px;border-radius:8px;font-size:12px;color:#9bb0c8}
</style>
</head>
<body>
<div id="hud">
  <b>__TITLE__</b><br/>rooms/zones: __ROOMS__ &nbsp;|&nbsp; objects: __ITEMS__ &nbsp;|&nbsp; depth: __DEPTH__
  <div class="modes">
    <button class="mode" id="m-walk">1 · Walk</button>
    <button class="mode active" id="m-orbit">2 · Orbit</button>
    <button class="mode" id="m-cine">3 · Cinematic</button>
  </div>
  <div id="hint">Walk: click to lock mouse · WASD move · Shift run · Space jump · Esc release<br/>
  Orbit: drag rotate · wheel zoom · right-drag pan<br/>
  Cinematic: camera trails the marker like a Remotion shot</div>
</div>
<div id="cross"></div>
<div id="label">loading __GLB__ …</div>
<script type="importmap">
{ "imports": {
  "three": "https://unpkg.com/three@0.160.0/build/three.module.js",
  "three/addons/": "https://unpkg.com/three@0.160.0/examples/jsm/"
} }
</script>
<script type="module">
import * as THREE from 'three';
import { OrbitControls } from 'three/addons/controls/OrbitControls.js';
import { GLTFLoader } from 'three/addons/loaders/GLTFLoader.js';
import { VRButton } from 'three/addons/webxr/VRButton.js';

const MAN = __MANIFEST__;
const AXIS = MAN.axis || 'z-up';
const toV = (a) => AXIS === 'z-up'
  ? new THREE.Vector3(a[0], a[2], -a[1])
  : new THREE.Vector3(a[0], a[1], a[2]);
const label = document.getElementById('label');
const cross = document.getElementById('cross');

const scene = new THREE.Scene();
scene.background = new THREE.Color(0x0e1116);
const camera = new THREE.PerspectiveCamera(60, innerWidth/innerHeight, 0.05, 500);
const renderer = new THREE.WebGLRenderer({ antialias:true });
renderer.setSize(innerWidth, innerHeight);
renderer.setPixelRatio(Math.min(devicePixelRatio, 2));
renderer.shadowMap.enabled = true;
renderer.shadowMap.type = THREE.PCFSoftShadowMap;
renderer.outputColorSpace = THREE.SRGBColorSpace;
renderer.xr.enabled = true;
document.body.appendChild(renderer.domElement);
document.body.appendChild(VRButton.createButton(renderer));

scene.add(new THREE.HemisphereLight(0xdfe8ff, 0x30343c, 0.9));
const sun = new THREE.DirectionalLight(0xffffff, 1.8);
sun.position.set(8, 16, 6);
sun.castShadow = true;
sun.shadow.mapSize.set(2048, 2048);
sun.shadow.bias = -0.0005;
scene.add(sun);
for (const z of MAN.zones || []) {
  const p = toV(z.position);
  const lamp = new THREE.PointLight(0xffe9c8, 0.7, 7);
  lamp.position.set(p.x, Math.min(p.y + 1.1, 2.6), p.z);
  scene.add(lamp);
}

const controls = new OrbitControls(camera, renderer.domElement);
controls.enableDamping = true;
controls.maxPolarAngle = Math.PI * 0.495;

// ---------- navigation grid (from the notebook manifest) ------------------
const nav = MAN.nav_grid;
function blocked(x, z) {
  if (!nav) return false;
  const ty = AXIS === 'z-up' ? -z : z;          // inverse of toV for the horizontal plane
  const i = Math.round((x - nav.min[0]) / nav.cell);
  const j = Math.round((ty - nav.min[1]) / nav.cell);
  if (i < 0 || j < 0 || i >= nav.cols || j >= nav.rows) return true;
  return nav.blocked[j][i] === '1';
}
const RAD = (MAN.physics && MAN.physics.radius_m) || 0.28;
const EYE = (MAN.physics && MAN.physics.eye_height_m) || 1.65;
function canStand(x, z, r=RAD) {
  const d = [[0,0],[r,0],[-r,0],[0,r],[0,-r],[r*.7,r*.7],[-r*.7,r*.7],[r*.7,-r*.7],[-r*.7,-r*.7]];
  for (const [dx,dz] of d) if (blocked(x+dx, z+dz)) return false;
  return true;
}
function nearestFree(x, z) {
  if (canStand(x,z)) return [x,z];
  for (let ring=1; ring<40; ring++) {
    for (let k=0; k<16; k++) {
      const a = k*Math.PI/8, nx=x+Math.cos(a)*ring*.12, nz=z+Math.sin(a)*ring*.12;
      if (canStand(nx,nz)) return [nx,nz];
    }
  }
  return [x,z];
}

// ---------- modes ----------------------------------------------------------
let mode = 'orbit';
let yaw = 0, pitch = 0, vy = 0, onGround = true;
const spawnV = toV((MAN.spawn && MAN.spawn.position) || [0,0,1.65]);
const sp = nearestFree(spawnV.x, spawnV.z);
const player = {x:sp[0], y:0, z:sp[1]};
yaw = (MAN.spawn && MAN.spawn.yaw) || 0;
const keys = Object.create(null);
addEventListener('keydown', e => {
  keys[e.code] = true;
  if (e.code==='Digit1') setMode('walk');
  if (e.code==='Digit2') setMode('orbit');
  if (e.code==='Digit3') setMode('cine');
  if (e.code==='Space' && mode==='walk' && onGround) { vy=4.2; onGround=false; }
});
addEventListener('keyup', e => { keys[e.code] = false; });
addEventListener('mousemove', e => {
  if (document.pointerLockElement !== renderer.domElement || mode!=='walk') return;
  yaw -= e.movementX * 0.0022;
  pitch = THREE.MathUtils.clamp(pitch - e.movementY*0.0022, -1.35, 1.35);
});
renderer.domElement.addEventListener('click', () => {
  if (mode==='walk' && document.pointerLockElement !== renderer.domElement)
    renderer.domElement.requestPointerLock();
});
function setMode(next) {
  mode = next;
  for (const m of ['walk','orbit','cine'])
    document.getElementById('m-'+m).classList.toggle('active', m===mode);
  controls.enabled = (mode==='orbit');
  cross.style.display = (mode==='walk') ? 'block' : 'none';
  if (mode==='walk') {
    const f = (MAN.spawn && MAN.spawn.position) || [0,0,1.65];
    const t = (MAN.zones && MAN.zones.length>1)
      ? MAN.zones[MAN.zones.length-1].position : [f[0], f[1]+2, f[2]];
    const a = toV(f), b = toV(t);
    yaw = Math.atan2(-(b.x-a.x), -(b.z-a.z));
    pitch = 0; vy = 0;
  }
  if (mode==='cine' && !curves) buildCurves();
  label.textContent = ({walk:'Walk mode · WASD + mouse', orbit:'Orbit overview',
                        cine:'Cinematic · trailing camera'})[mode];
}
document.getElementById('m-walk').onclick = () => setMode('walk');
document.getElementById('m-orbit').onclick = () => setMode('orbit');
document.getElementById('m-cine').onclick = () => setMode('cine');

// ---------- cinematic trailing-camera paths --------------------------------
let curves = null, marker = null, lookAt = new THREE.Vector3();
function buildCurves() {
  const cp = MAN.camera_paths;
  if (!cp || !cp.camera || cp.camera.length < 3) return;
  const subj = cp.subject.map(k => toV(k.position));
  const cam  = cp.camera.map(k => toV(k.position));
  curves = {
    subject: new THREE.CatmullRomCurve3(subj, true, 'centripetal'),
    camera:  new THREE.CatmullRomCurve3(cam, true, 'centripetal'),
    frames: (MAN.camera_settings && MAN.camera_settings.frames) || 300
  };
  marker = new THREE.Mesh(new THREE.SphereGeometry(0.13, 16, 12),
    new THREE.MeshStandardMaterial({color:0xffb24a, emissive:0x6b3300}));
  scene.add(marker);
  camera.position.copy(curves.camera.getPointAt(0));
  lookAt.copy(curves.subject.getPointAt(0));
}
function cinematic(frame) {
  if (!curves) return;
  const t = (frame % curves.frames) / curves.frames;
  const lag = MAN.camera_settings.trailing.look_at_lag_frames || 9;
  const tl = ((frame - lag + curves.frames) % curves.frames) / curves.frames;
  const subj = curves.subject.getPointAt(t);
  const goal = curves.camera.getPointAt(t);
  const target = curves.subject.getPointAt(tl).add(new THREE.Vector3(0,0.6,0));
  const smooth = MAN.camera_settings.trailing.smoothing || 0.14;
  camera.position.lerp(goal, smooth);
  lookAt.lerp(target, smooth);
  camera.lookAt(lookAt);
  if (marker) marker.position.copy(subj);
}

new GLTFLoader().load('__GLB__', (gltf) => {
  const root = gltf.scene;
  if (AXIS === 'z-up') root.rotation.x = -Math.PI/2;   // trimesh Z-up -> three.js Y-up
  root.traverse(o => { if (o.isMesh) {
    o.castShadow = true; o.receiveShadow = true;
    const mats = Array.isArray(o.material) ? o.material : [o.material];
    mats.forEach(m => { if (m) m.side = THREE.DoubleSide; });
  }});
  scene.add(root);
  const box = new THREE.Box3().setFromObject(root);
  const size = box.getSize(new THREE.Vector3());
  const center = box.getCenter(new THREE.Vector3());
  controls.target.copy(center);
  const r = Math.max(size.x, size.y, size.z);
  camera.position.copy(center).add(new THREE.Vector3(r*0.9, r*0.75, r*1.1));
  camera.near = r/200; camera.far = r*50; camera.updateProjectionMatrix();
  controls.update();
  label.textContent = ({walk:'Walk mode · click to look around', orbit:'Orbit overview',
                        cine:'Cinematic · trailing camera'})[mode];
}, undefined, (err) => { console.error(err); label.textContent = 'GLB load failed: '+err.message; });

addEventListener('resize', () => {
  camera.aspect = innerWidth/innerHeight; camera.updateProjectionMatrix();
  renderer.setSize(innerWidth, innerHeight);
});

let frameNo = 0;
function loop() {
  frameNo++;
  if (!renderer.xr.isPresenting) {
    if (mode==='orbit') {
      controls.update();
    } else if (mode==='walk') {
      const run = keys.ShiftLeft || keys.ShiftRight ? 1.8 : 1.0;
      const speed = 2.8 * run;
      const fx = -Math.sin(yaw), fz = -Math.cos(yaw);
      const rx = Math.cos(yaw), rz = -Math.sin(yaw);
      let mx=0, mz=0;
      if (keys.KeyW || keys.ArrowUp)    { mx+=fx; mz+=fz; }
      if (keys.KeyS || keys.ArrowDown)  { mx-=fx; mz-=fz; }
      if (keys.KeyD || keys.ArrowRight) { mx+=rx; mz+=rz; }
      if (keys.KeyA || keys.ArrowLeft)  { mx-=rx; mz-=rz; }
      const ln = Math.hypot(mx,mz) || 1;
      mx = mx/ln*speed/60; mz = mz/ln*speed/60;
      if (canStand(player.x+mx, player.z)) player.x += mx;
      if (canStand(player.x, player.z+mz)) player.z += mz;
      vy -= 11.0/60;
      player.y = Math.max(0, player.y + vy/60);
      if (player.y === 0) { vy = 0; onGround = true; }
      camera.position.set(player.x, player.y + EYE, player.z);
      camera.rotation.order = 'YXZ';
      camera.rotation.y = yaw; camera.rotation.x = pitch;
    } else if (mode==='cine') {
      cinematic(frameNo);
    }
  }
  renderer.render(scene, camera);
}
setMode('orbit');
renderer.setAnimationLoop(loop);
</script>
</body>
</html>"""

for i, (glb, meta) in enumerate(exports, 1):
    payload = json.dumps(meta, separators=(",", ":")).replace("</", "<\\/")
    html = (VIEWER.replace("__TITLE__", f"Environment {i}")
                  .replace("__GLB__", glb.name)
                  .replace("__ROOMS__", str(len(meta["rooms"]) or len(meta["zones"])))
                  .replace("__ITEMS__", str(len(meta["furniture"])))
                  .replace("__DEPTH__", str(meta.get("depth_method") or "n/a"))
                  .replace("__MANIFEST__", payload))
    (OUT / "web" / f"viewer_{i}.html").write_text(html)
    print(f"  wrote viewer_{i}.html  ->  loads {glb.name}  (walk / orbit / cinematic + VR)")

## 📊 9 · Visualize + summary

In [ ]:
def visualize(sb, scene_id):
    if sb.get("kind") == "photo":
        src = cv2.cvtColor(cv2.imread(sb["plan"]["path"]), cv2.COLOR_BGR2RGB)
        dep = cv2.imread(str(OUT / "visualizations" / "depth.png"), cv2.IMREAD_GRAYSCALE)
        fig, (a1, a2) = plt.subplots(1, 2, figsize=(15, 6))
        a1.imshow(src); a1.set_title(f"Scene {scene_id}: source image"); a1.axis("off")
        a2.imshow(dep, cmap="magma"); a2.set_title("estimated depth (near = bright)"); a2.axis("off")
        plt.tight_layout()
        p = OUT / "visualizations" / f"scene_{scene_id}.png"
        fig.savefig(p, dpi=110, bbox_inches="tight"); plt.close(fig)
        return p
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    for r in sb["rooms"]:
        minx, miny, maxx, maxy = r["bounds"]
        ax1.add_patch(Rectangle((minx, miny), maxx - minx, maxy - miny,
                                ec="black", fc="#eaf2ff", alpha=0.9, lw=1.4))
        ax1.text((minx + maxx) / 2, (miny + maxy) / 2, r["type"],
                 ha="center", va="center", fontsize=7)
    for it in sb["furniture"]:
        x, y = it["position"][0], it["position"][1]
        w, d = it["size"][0], it["size"][1]
        ax1.add_patch(Rectangle((x, y), w, d, ec="none", fc="#d98b2b", alpha=0.85))
    ax1.set_aspect("equal"); ax1.set_title(f"Scene {scene_id}: plan (top view)")
    ax1.set_xlabel("x (m)"); ax1.set_ylabel("y (m)"); ax1.grid(alpha=0.25)

    ax2 = fig.add_subplot(122, projection="3d")
    shown = 0
    for kind, _rid, m in sb["meshes"]:
        if kind == "wall":
            continue
        v, f = m.vertices, m.faces
        try:
            ax2.plot_trisurf(v[:, 0], v[:, 1], v[:, 2], triangles=f,
                             shade=True, cmap="cividis", linewidth=0, alpha=0.95)
            shown += 1
        except Exception:
            pass
        if shown > 18:
            break
    ax2.set_title(f"Scene {scene_id}: 3D (floors + furniture)")
    ax2.set_box_aspect((1, 1, 0.4))
    ax2.set_xlabel("x"); ax2.set_ylabel("y"); ax2.set_zlabel("z")
    plt.tight_layout()
    p = OUT / "visualizations" / f"scene_{scene_id}.png"
    fig.savefig(p, dpi=110, bbox_inches="tight"); plt.close(fig)
    return p

rows = []
for i, sb in enumerate(scene_builds, 1):
    try:
        visualize(sb, i)
    except Exception as e:
        print("  (visualization skipped:", type(e).__name__, ")")
    rows.append({"scene": i, "kind": sb.get("kind", "plan"),
                 "source": Path(sb["plan"]["path"]).name,
                 "rooms": len(sb["rooms"]), "furniture": len(sb["furniture"]),
                 "meshes": len(sb["meshes"])})
print("Summary:")
print(pd.DataFrame(rows).to_string(index=False))


## 💾 10 · Package results

In [ ]:
README = OUT / "README.md"
README.write_text(
    "# Explorable 3D World from a 2D Image\n\n"
    "Generated by `ar_vr_environment_kaggle.ipynb`.\n\n"
    "- `3d/scene_*.glb`   single-file binary glTF with PBR materials + embedded textures (Z-up)\n"
    "- `3d/gltf/`         JSON glTF + external buffers (interchange)\n"
    "- `3d/scene_*.json`  spawn, zones, nav grid, furniture, lights, camera settings\n"
    "- `3d/camera_path_*.json`  looping subject + trailing-camera keyframes for Remotion (Z-up)\n"
    "- `textures/*.png`   the procedural textures used (plan path)\n"
    "- `visualizations/*.png`  plan + 3D previews (photo path: source + depth)\n"
    "- `web/viewer_*.html`  three.js viewer: Walk (WASD + collision), Orbit, Cinematic trailing camera, WebXR\n\n"
    "Plan detail: extruded walls with real door gaps, per-room floors, baseboards,\n"
    "rugs, ceiling lights, and multi-part furniture. Photo detail: a walkable room shell\n"
    "with the image as a depth-displaced backdrop/floor/ceiling and image-derived props.\n\n"
    "Viewer: serve `outputs/` over HTTP and open `web/viewer_1.html`.\n"
    "Keys: 1 walk, 2 orbit, 3 cinematic; click to lock the mouse in walk mode.\n"
    "Camera data is in trimesh Z-up: convert with (x, y, z) -> (x, z, -y).\n\n"
    "Targets: WebXR / three.js / Remotion / Babylon, Unity, Unreal, ARKit, ARCore.\n"
)
zip_path = WORKDIR / "environments.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in OUT.rglob("*"):
        if f.is_file():
            z.write(f, f.relative_to(OUT))
print("ZIP:", zip_path, f"({zip_path.stat().st_size/1024:.0f} KB)")
print("Files under /kaggle/working are saved as the notebook output automatically.")
print()
print("Done. Serve outputs/ over HTTP and open viewer_1.html: walk the world, or use "
      "camera_path_1.json for a Remotion trailing-camera shot.")